# *TO-DO*: New model grid

In [ ]:
n_age_bins = 6
np.geomspace(0.1, 10, n_age_bins+1)[:-1]

In [ ]:
n_birth_steps = 10
np.geomspace(0.01, 10, n_birth_steps)

In [ ]:
n = 49
exponent = 1
for x0 in np.linspace(1, n, n)/(n+1):
    x = np.concatenate([x0 * np.power(np.linspace(1, n, n)/(n+1), exponent), 1 - (1-x0) * np.power(np.linspace(n, 1, n)/(n+1), exponent)])
    plt.plot(x, x0*np.ones_like(x), 'k.', alpha=.2)
plt.xlim(0, 1)
plt.ylim(0, 1)
#plt.plot([0, 1], [0, 1])
plt.ylabel("M/$M_0$")
plt.xlabel("t/$t_0$")
print(f'{n} parameters, {np.power(float(2*n), n):.1e} potential models')

# Initialisation

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors
from astropy.table import Table, join
from astropy.cosmology import Planck18
from astropy import units as u

## Read data

In [ ]:
specz = join(Table.read('data/BESTA//mer_phz_specz_sample.fits', unit_parse_strict='silent'),
             Table.read('data/BESTA/besta_results_specz.fits'),
             keys='OBJECT_ID')

In [ ]:
photoz = join(Table.read('data/BESTA//mer_phz_specz_sample.fits', unit_parse_strict='silent'),
              Table.read('data/BESTA/besta_results_photoz.fits'),
              keys='OBJECT_ID')

In [ ]:
besta_dr1 = Table.read("data/BESTA/stacked_catalogue.fits", unit_parse_strict='silent')

## Redshift bins

In [ ]:
redshift_bins = np.linspace(0.05, 0.55, 6)
redshift_centre = (redshift_bins[:-1] + redshift_bins[1:]) / 2
redshift_colour = ['cyan', 'blue', 'green', 'orange', 'red']
redshift_labels = [f'z={z0:.2f}' for z0 in redshift_centre]

In [ ]:
zz = np.linspace(redshift_bins[0], redshift_bins[-1], 101)
cosmic_time = Planck18.age(zz)
main_sequence = -np.log10(cosmic_time.to_value(u.yr))

In [ ]:
plt.plot(zz, cosmic_time.to_value(u.Gyr), 'k-')
for edge in redshift_bins:
    plt.axvline(edge, c='k', ls=':')
plt.ylabel("cosmic time [Gyr]")
plt.xlabel("redshift")
plt.grid(alpha=.2)

# Spectroscopic sample

In [ ]:
redshift_bins = np.linspace(0.05, 0.55, 6)
redshift_centre = (redshift_bins[:-1] + redshift_bins[1:]) / 2
redshift_colour = ['cyan', 'blue', 'green', 'orange', 'red']
redshift_labels = [f'z={z0:.2f}' for z0 in redshift_centre]

In [ ]:
zz = np.linspace(.1*redshift_bins[0], redshift_bins[-1], 101)
cosmic_time = Planck18.age(zz)
main_sequence = -np.log10(cosmic_time.to_value(u.yr))

In [ ]:
log_mass_threshold_z1 = 11.5
log_mass_massive = 11

## Best fit $\chi^2$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True, gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_xlabel('best-fit $\chi^2$')
ax.hist(specz['bestfit_chi2'], bins=np.linspace(0, 11, 101), color='k', alpha=1, label='total', density=True, histtype='step')
x = []
for i, z0 in enumerate(redshift_centre):
    galaxies = np.where((specz['Z'] > redshift_bins[i]) & (specz['Z'] <= redshift_bins[i+1]))
    x.append(specz['bestfit_chi2'][galaxies])
ax.hist(x[::-1], bins=np.linspace(0, 11, 101), histtype='bar', stacked=True, color=redshift_colour[::-1], label=redshift_labels[::-1], density=True)
ax.legend()

ax = axes[0, 1]
ax.set_title('photoz')
#ax.yaxis.set_ticklabels('')
ax.set_xlabel('best-fit $\chi^2$')
ax.hist(photoz['bestfit_chi2'], bins=np.linspace(0, 11, 101), color='k', alpha=1, label='total', density=True, histtype='step')
x = []
for i, z0 in enumerate(redshift_centre):
    galaxies = np.where((photoz['PHZ_PP_MEDIAN_REDSHIFT'] > redshift_bins[i]) & (photoz['PHZ_PP_MEDIAN_REDSHIFT'] <= redshift_bins[i+1]))
    x.append(photoz['bestfit_chi2'][galaxies])
ax.hist(x[::-1], bins=np.linspace(0, 11, 101), histtype='bar', stacked=True, color=redshift_colour[::-1], label=redshift_labels[::-1], density=True)
ax.legend()

ax = axes[0, 2]
ax.set_title('DR1')
#ax.yaxis.set_ticklabels('')
ax.set_xlabel('best-fit $\chi^2$')
ax.hist(besta_dr1['bestfit_chi2'], bins=np.linspace(0, 11, 101), color='k', alpha=1, label='total', density=True, histtype='step')
x = []
for i, z0 in enumerate(redshift_centre):
    galaxies = np.where((besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'] > redshift_bins[i]) & (besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'] <= redshift_bins[i+1]))
    x.append(besta_dr1['bestfit_chi2'][galaxies])
ax.hist(x[::-1], bins=np.linspace(0, 11, 101), histtype='bar', stacked=True, color=redshift_colour[::-1], label=redshift_labels[::-1], density=True)
ax.legend()
#ax.set_yscale('log')

## Photometric redshift

In [ ]:
def normalised_error(a, b):
    return (a - b) / (np.abs(a) + np.abs(b))

specz_error = normalised_error(specz['PHZ_PP_MEDIAN_REDSHIFT'], specz['Z'])
photoz_error = normalised_error(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['Z'])

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10), width_ratios=(1, 1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.Normalize(vmin=-.35, vmax=.35)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_ylabel('PHZ median z')
ax.set_xlabel('z')
ax.scatter(specz['Z'], specz['PHZ_PP_MEDIAN_REDSHIFT'], c = specz_error, norm=norm, cmap=cmap, s=1, alpha=.5)
ax.plot([0, 2], [0, 2], 'k:')

ax = axes[0, 1]
ax.set_title('photoz')
ax.yaxis.set_ticklabels('')
ax.set_xlabel('z')
ax.scatter(photoz['Z'], photoz['PHZ_PP_MEDIAN_REDSHIFT'], c = photoz_error, norm=norm, cmap=cmap, s=1, alpha=.5)
ax.plot([0, 2], [0, 2], 'k:')

plt.colorbar(cm, cax=axes[0, 2], label="$\Delta z \equiv$ (PHZ - Z) / (|PHZ| + |Z|)")


norm = colors.Normalize(vmin=0, vmax=3.75)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel('PHZ median z')
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['PHZ_PP_MEDIAN_REDSHIFT'], c = specz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.5)
ax.plot([0, 2], [0, 2], 'k:')

ax = axes[1, 1]
ax.yaxis.set_ticklabels('')
ax.set_xlabel('z')
ax.scatter(photoz['Z'], photoz['PHZ_PP_MEDIAN_REDSHIFT'], c = photoz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.5)
ax.plot([0, 2], [0, 2], 'k:')

plt.colorbar(cm, cax=axes[1, 2], label="best-fit $\chi^2$")


norm = colors.Normalize(vmin=-.35, vmax=.35)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[2, 0]
ax.set_ylabel("best-fit $\chi^2$")
ax.set_ylim(0, 9.75)
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['bestfit_chi2'], c = specz_error, norm=norm, cmap=cmap, s=1, alpha=.5)

ax = axes[2, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(0, 9.75)
ax.set_xlabel('spectroscopic z')
ax.scatter(photoz['Z'], photoz['bestfit_chi2'], c = photoz_error, norm=norm, cmap=cmap, s=1, alpha=.5)

plt.colorbar(cm, cax=axes[2, 2], label="$\Delta z \equiv$ (PHZ - Z) / (|PHZ| + |Z|)")


for ax in axes[:, :-1].ravel():
    for edge in redshift_bins:
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(0, redshift_bins[-1])

for ax in axes[:-1, :-1].ravel():
    ax.set_ylim(0, redshift_bins[-1])
    ax.plot([0, 2], [0, 2], 'k:')


In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 4), sharex=True, sharey=True, gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

bins = np.linspace(0, redshift_bins[-1], 101)

ax = axes[0, 0]
ax.hist(specz['Z'], bins=bins, color='c', alpha=.25, label='specz')
ax.hist(specz['Z'], bins=bins, color='c', histtype='step')

ax.hist(photoz['PHZ_PP_MEDIAN_REDSHIFT'], bins=bins, color='r', alpha=.25, label='photoz')
ax.hist(photoz['PHZ_PP_MEDIAN_REDSHIFT'], bins=bins, color='r', histtype='step')

ax.hist(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], bins=bins, color='k', alpha=.25, label='DR1')
ax.hist(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], bins=bins, color='k', histtype='step')

ax.legend()
ax.set_xlabel("PHZ median z")
ax.set_yscale('log')

for i, edge in enumerate(redshift_bins):
    ax.axvline(edge, c='k', ls=':')

# M(z) and sSFR(z)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8), width_ratios=(1, 1, 1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.Normalize(vmin=0, vmax=3.75)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_ylabel('log ( M / M$_\odot$ )')
ax.set_ylim(8.5, 12.5)
ax.scatter(specz['Z'], specz['stellar_mass__mean'], c=specz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 1]
ax.set_title('photoz')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['stellar_mass__mean'], c=photoz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 2]
ax.set_title('DR1')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], c=besta_dr1['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.05)

plt.colorbar(cm, cax=axes[0, -1], label="best-fit $\chi^2$")


norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['log_ssfr_8p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_8p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['log_ssfr_9p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_9p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(-.01, redshift_bins[-1])

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8), width_ratios=(1, 1, 1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.Normalize(vmin=0, vmax=3.75)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_ylabel('log ( M / M$_\odot$ )')
ax.set_ylim(8.5, 12.5)
ax.scatter(specz['Z'], specz['stellar_mass__mean'], c=specz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 1]
ax.set_title('photoz')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['stellar_mass__mean'], c=photoz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 2]
ax.set_title('DR1')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], c=besta_dr1['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[0, -1], label="best-fit $\chi^2$")


norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['log_ssfr_8p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_8p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
ax.scatter(specz['Z'], specz['log_ssfr_9p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_9p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(-.01, 4*redshift_bins[-1])

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

## Mass function

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8), gridspec_kw={'hspace': 0, 'wspace': 0}, squeeze=False, sharex=True, sharey=True)

def plot_distributions(plot_column, title, table, redshit_column):
    ax = axes[:, plot_column]
    ax[0].set_title(title)
    for i, z0 in enumerate(redshift_centre):
        galaxies = np.where(
            (table[redshit_column] > redshift_bins[i])
            & (table[redshit_column] <= redshift_bins[i+1])
            & (table['stellar_mass__mean'] > log_mass_threshold_z1 + 2*np.log10(redshift_bins[i+1]))
        )
        log_mass = np.sort(table['stellar_mass__mean'][galaxies])
        mass_above = np.cumsum(10**log_mass[::-1])
        n11 = log_mass.size - np.searchsorted(log_mass, log_mass_massive)
        ax[0].plot(log_mass[::-1], np.arange(log_mass.size)/n11, label=f'z={z0:.2f}', c=redshift_colour[i])
        ax[1].plot(log_mass[::-1], mass_above/mass_above[n11], label=f'z={z0:.2f}', c=redshift_colour[i])
        ax[0].legend()
        if plot_column == 0:
            ax[0].set_ylabel(f"N(M) / N($10^{{{log_mass_massive}}}$ M$_\odot$)")
            ax[1].set_ylim(3e-4, 3e2)
            ax[0].set_yscale('log')
            ax[1].set_ylabel(f"M(M) / M($10^{{{log_mass_massive}}}$ M$_\odot$)")
            ax[1].set_yscale('log')
        ax[1].set_xlabel("log( M / M$_\odot$ )")
        ax[1].set_xlim(9.75, 12.25)
        ax[0].grid(alpha=.2)
        ax[1].grid(alpha=.2)

plot_distributions(0, 'specz', specz, 'Z')
plot_distributions(1, 'photoz', photoz, 'PHZ_PP_MEDIAN_REDSHIFT')
plot_distributions(2, 'DR1', besta_dr1, 'PHZ_PP_MEDIAN_REDSHIFT')

# Main sequence

In [ ]:
n_bins = redshift_centre.size
norm = colors.Normalize(vmin=-.75, vmax=1.5)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

def plot_main_sequence(joint_table, redshift_column):
    fig, axes = plt.subplots(2, n_bins, sharey=True, sharex=True, figsize=(15, 4), gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)
    for i, z0 in enumerate(redshift_centre):
        galaxies = np.where((joint_table[redshift_column] > redshift_bins[i]) & (joint_table[redshift_column] <= redshift_bins[i+1]))
        mass = joint_table['stellar_mass__mean'][galaxies]
        ssfr8 = joint_table['log_ssfr_8p0__mean'][galaxies]
        ssfr9 = joint_table['log_ssfr_9p0__mean'][galaxies]
        ax = axes[0, i]
        sc = ax.scatter(mass, ssfr8, s=1, alpha=.5, c=ssfr9-ssfr8, norm=norm, cmap=cmap)
        ax.axvline(11.5+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='b', ls='--', label=f'z={z0:.2f}')
        ax = axes[1, i]
        sc = ax.scatter(mass, ssfr9, s=1, alpha=.5, c=ssfr9-ssfr8, norm=norm, cmap=cmap)
        ax.axvline(11.5+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='b', ls='--', label=f'z={z0:.2f}')
        ax.legend(loc='lower left')
    ax.set_xlim(9.7, 11.8)
    ax.set_ylim(-13.5, -8.5)
    axes[0, 0].set_ylabel("log( sSFR8 / yr )")
    axes[1, 0].set_ylabel("log( sSFR9 / yr )")
    axes[1, 2].set_xlabel("log( M / M$_\odot$ )")
    cb = plt.colorbar(cm, ax=axes[:, -1], label='log( sSFR9 / sSFR8 )')

In [ ]:
plot_main_sequence(specz, 'Z')

In [ ]:
plot_main_sequence(photoz, "PHZ_PP_MEDIAN_REDSHIFT")

In [ ]:
plot_main_sequence(besta_dr1, "PHZ_PP_MEDIAN_REDSHIFT")

## Main sequence

In [ ]:
norm = colors.Normalize(vmin=-.75, vmax=1.5)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
for i, z0 in enumerate(redshift_centre):
    #galaxies = np.where((joint_table['Z'] > redshift_bins[i]) & (joint_table['Z'] <= redshift_bins[i+1]))
    galaxies = np.where((joint_table['PHZ_PP_MEDIAN_REDSHIFT'] > redshift_bins[i]) & (joint_table['Z'] <= redshift_bins[i+1]))
    mass = joint_table['stellar_mass__mean'][galaxies]
    ssfr8 = joint_table['log_ssfr_8p0__mean'][galaxies]
    ssfr9 = joint_table['log_ssfr_9p0__mean'][galaxies]
    ax = axes[0, i]
    sc = ax.scatter(mass, ssfr8, s=1, alpha=.5, c=ssfr9-ssfr8, norm=norm, cmap=cmap)
    ax.axvline(11.5+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
    ax.axhline(np.interp(z0, zz, main_sequence), c='b', ls='--', label=f'z={z0:.2f}')
    ax = axes[1, i]
    sc = ax.scatter(mass, ssfr9, s=1, alpha=.5, c=ssfr9-ssfr8, norm=norm, cmap=cmap)
    ax.axvline(11.5+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
    ax.axhline(np.interp(z0, zz, main_sequence), c='b', ls='--', label=f'z={z0:.2f}')
    ax.legend(loc='upper left')
ax.set_xlim(9.7, 11.8)
ax.set_ylim(-13.5, -8.5)
axes[0, 0].set_ylabel("log( sSFR8 / yr )")
axes[1, 0].set_ylabel("log( sSFR9 / yr )")
axes[1, 2].set_xlabel("log( M / M$_\odot$ )")
cb = plt.colorbar(cm, ax=axes[:, -1], label='log( sSFR9 / sSFR8 )')

*TO-DO*:
- Add mass-weighted age
- Compare spec and photo $z$

In [ ]:
plt.hist(ssfr8-ssfr9, bins=100)

# Old tests

## MAP, median, mean
I understand the the Maximum A Posteriori (MAP) follows a discrete grid

In [ ]:
plt.plot(joint_table['log_ssfr_9p0__map'], joint_table['log_ssfr_8p0__map'], 'c,', alpha=.25)

However, median and percentiles should interpolate, shouldn't they?

In [ ]:
plt.plot(joint_table['log_ssfr_9p0__median'], joint_table['log_ssfr_8p0__median'], 'c,', alpha=.25)

In [ ]:
plt.plot(joint_table['log_ssfr_9p0__hi68'], joint_table['log_ssfr_8p0__hi68'], 'c,', alpha=.25)

The mean seems to be more useful

In [ ]:
plt.plot(joint_table['log_ssfr_9p0__mean'], joint_table['log_ssfr_8p0__mean'], 'c,', alpha=.25)

## Scaling relations

In [ ]:
fig = plt.figure('SFMS')
ax = fig.subplots(2)

sc = ax[0].scatter(joint_table['stellar_mass__mean'], joint_table['log_ssfr_9p0__mean'], s=1, alpha=.1, c=joint_table['bestfit_chi2'], norm=colors.Normalize(vmax=3), cmap="seismic")
sc = ax[1].scatter(joint_table['stellar_mass__mean'], joint_table['log_ssfr_8p0__mean'], s=1, alpha=.1, c=joint_table['bestfit_chi2'], norm=colors.Normalize(vmax=3), cmap="seismic")
plt.colorbar(sc, ax=ax[1], alpha=1, orientation='horizontal')

In [ ]:
fig = plt.figure('SFMS')
ax = fig.subplots(2)

ax[0].plot(joint_table['stellar_mass__mean'], joint_table['log_ssfr_9p0__mean'], 'r,', alpha=.1)
ax[0].plot(joint_table['stellar_mass__mean'][good], joint_table['log_ssfr_9p0__mean'][good], 'k,', alpha=.25)
ax[1].plot(joint_table['stellar_mass__mean'], joint_table['log_ssfr_8p0__mean'], 'r,', alpha=.1)
ax[1].plot(joint_table['stellar_mass__mean'][good], joint_table['log_ssfr_8p0__mean'][good], 'k,', alpha=.25)

In [ ]:
plt.scatter(joint_table['stellar_mass__mean'], joint_table['z_ism_today__mean'], s=1, alpha=.1, c=joint_table['bestfit_chi2'], norm=colors.Normalize(vmax=3), cmap="seismic")

In [ ]:
plt.plot(joint_table['stellar_mass__mean'], joint_table['z_ism_today__mean'], 'r,', alpha=.1)
plt.plot(joint_table['stellar_mass__mean'][good], joint_table['z_ism_today__mean'][good], 'k,', alpha=.25)